 ## Extract CMVR Rules, Rule Text, and AIS References into MongoDB



This notebook reuses the project's verified Docling parsing and regulatory-context pipeline to build one canonical MongoDB document per CMVR rule.



Each document stores the complete normalized `ruleText`, including sub-rules, provisos, tables, footnotes, and annex material assigned to that rule. The LLM returns AIS references as a strict list of closed objects for OpenAI Structured Outputs compatibility; the notebook converts that list into the requested MongoDB object.



```json

{

  "_id": "cmvr-1989:rule:115",

  "canonicalKey": "cmvr-1989:rule:115",

  "canonicalTitle": "Emission of smoke, vapour, etc",

  "chapterId": "cmvr-1989:chapter:V",

  "documentId": "cmvr-1989",

  "ruleNumber": "115",

  "ruleText": "115. Emission of smoke, vapour, etc. ...",

  "status": "active",

  "AIS": {

    "AIS-025": "Source-grounded AIS description"

  }

}

```



Paid OpenAI calls are limited to AIS-bearing rules. MongoDB writes remain controlled by `WRITE_TO_MONGODB`.

## 1. Configure Document and MongoDB Settings



Import dependencies, load `.env`, and define source, LLM, checkpoint, and MongoDB settings. Leave `RUN_LLM_EXTRACTION` and `WRITE_TO_MONGODB` set to `False` until previews and validation look correct.

In [63]:
from __future__ import annotations



import json

import logging

import os

import re

import unicodedata

from collections import Counter, defaultdict

from dataclasses import dataclass

from datetime import datetime, timezone

from pathlib import Path

from typing import Any



import pandas as pd

from dotenv import load_dotenv

from openai import OpenAI

from openai.lib._pydantic import to_strict_json_schema

from pydantic import BaseModel, ConfigDict, Field, ValidationError, model_validator

from pymongo import ASCENDING, MongoClient, UpdateOne

from pymongo.errors import BulkWriteError, PyMongoError



from ingest_regulatory_graph import (

    DocumentKind,

    RegulatoryChunk,

    build_converter,

    chunk_overlaps_page_range,

    convert_document,

    iter_regulatory_chunks,

)



load_dotenv(Path(".env"))

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

LOGGER = logging.getLogger("cmvr_rule_notebook")



# Document and parsing

PDF_PATH = Path("cmvr-1989.pdf").resolve()

DOCUMENT_ID = "cmvr-1989"

DOCUMENT_NAME = "Central Motor Vehicles Rules, 1989"

PAGE_RANGE: tuple[int, int] | None = None  # Example: (95, 122); None means all pages.

ENABLE_OCR = False

MAX_CHUNK_CHARS = 12_000

CHUNK_OVERLAP = 300

MAX_RULES: int | None = None  # Use a small integer for a paid validation run.



# LLM extraction

RUN_LLM_EXTRACTION = False

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

OPENAI_TIMEOUT_SECONDS = 120.0

OPENAI_MAX_RETRIES = 4

MAX_RULE_INPUT_CHARS = 80_000

AIS_CONTEXT_RADIUS = 1_500



# Checkpoint and review artifacts

CHECKPOINT_PATH = Path("cmvr_rule_ais_checkpoint.jsonl")

REVIEW_DIR = Path("cmvr_rule_review")



# MongoDB

WRITE_TO_MONGODB = True

MONGODB_URI = os.getenv("MONGODB_URI")

MONGODB_DATABASE = os.getenv("MONGODB_DATABASE", "automotive_regulations_extract")

MONGODB_COLLECTION = os.getenv("CMVR_RULE_COLLECTION", "cmvr_rules")

MONGO_BATCH_SIZE = 100



assert PDF_PATH.is_file(), f"CMVR PDF not found: {PDF_PATH}"

assert DOCUMENT_ID == DOCUMENT_ID.lower(), "DOCUMENT_ID must be lowercase"

print(f"PDF: {PDF_PATH.name}")

print(f"OpenAI configured: {bool(OPENAI_API_KEY)}")

print(f"MongoDB configured: {bool(MONGODB_URI)}")

print(f"Paid extraction enabled: {RUN_LLM_EXTRACTION}")

print(f"MongoDB writes enabled: {WRITE_TO_MONGODB}")

PDF: cmvr-1989.pdf
OpenAI configured: True
MongoDB configured: True
Paid extraction enabled: False
MongoDB writes enabled: True


## 2. Load the CMVR PDF Using the Existing Extraction Pipeline



Use the same `DocumentConverter`, `HierarchicalChunker`, page provenance, table serialization, and repaired chapter/rule context used by `ingest_regulatory_graph.py`. For a page range, pages before the selected range are still parsed to recover chapter context, but only overlapping chunks are retained.

In [64]:
conversion_page_range = (1, PAGE_RANGE[1]) if PAGE_RANGE else None

converter = build_converter(enable_ocr=ENABLE_OCR)

docling_document = convert_document(converter, PDF_PATH, conversion_page_range)



regulatory_chunks = list(

    iter_regulatory_chunks(

        docling_document,

        document_kind=DocumentKind.CMVR,

        max_chunk_chars=MAX_CHUNK_CHARS,

        chunk_overlap=CHUNK_OVERLAP,

    )

)

if PAGE_RANGE:

    regulatory_chunks = [

        chunk

        for chunk in regulatory_chunks

        if chunk_overlaps_page_range(chunk, PAGE_RANGE)

    ]



print(f"Contextual chunks retained: {len(regulatory_chunks):,}")

print(f"Source pages represented: {len({page for chunk in regulatory_chunks for page in chunk.page_numbers}):,}")

pd.DataFrame(

    [

        {

            "pages": ", ".join(map(str, chunk.page_numbers)),

            "chapter": chunk.chapter,

            "rule": chunk.rule,

            "contentType": chunk.content_type,

            "text": chunk.text[:180],

        }

        for chunk in regulatory_chunks[:10]

    ]

)

INFO: detected formats: [<InputFormat.PDF: 'pdf'>]
INFO: Going to convert document batch...
INFO: Initializing pipeline for StandardPdfPipeline with options hash 49f9156e34a5c42febf31f3440a56588
INFO: Accelerator device: 'mps'
INFO: HTTP Request: GET https://huggingface.co/api/models/docling-project/docling-layout-heron/revision/main "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 770/770 [00:00<00:00, 1385.19it/s]
INFO: HTTP Request: GET https://huggingface.co/api/models/docling-project/docling-models/revision/v2.3.0 "HTTP/1.1 200 OK"
INFO: Accelerator device: 'mps'
INFO: Processing document cmvr-1989.pdf
INFO: Finished converting document cmvr-1989.pdf in 54.77 sec.


Contextual chunks retained: 1,716
Source pages represented: 174


,pages,chapter,rule,contentType,text
0,1,CHAPTER I: PRELIMINARY,Rule 1 > Sub-rule (1) > Proviso,text,1. Short title and commencement.-(1) The...
1,1,CHAPTER I: PRELIMINARY,Rule 2,text,"2. Definitions.-In these rules, unless the con..."
2,"1, 2",CHAPTER I: PRELIMINARY,NaN,text,Explanation. -A construction equipment vehi...
3,1,CHAPTER I: PRELIMINARY,Clause 1,text,"1 Vide G.S.R. 590 (E), dated 2-6-1989, publish..."
4,1,CHAPTER I: PRELIMINARY,Clause 2,text,"2 Inserted by G.S.R. 933(E), dated 28-10-1989 ..."
5,1,CHAPTER I: PRELIMINARY,Clause 3,text,"3 The words and figures ""sub-rule (3) of rule ..."
6,1,CHAPTER I: PRELIMINARY,Clause 4,text,"4 Inserted by G.S.R. 338(E), dated 26-3-1993 (..."
7,1,CHAPTER I: PRELIMINARY,Clause 5,text,"5 Inserted by G.S.R. 642(E), dated 28-7-2000 (..."
8,2,CHAPTER I: PRELIMINARY,Clause 6,text,"6 [ 7 (d)] ""financier"" means a person or a tit..."
9,2,CHAPTER I: PRELIMINARY,Clause 8,text,"8 [(e)] ""Form"" means a Form appended to these ..."


## 3. Normalize Extracted PDF Text



Normalize OCR and layout artifacts without removing rule numbers, sub-rule markers, table values, or AIS identifiers. Docling remains responsible for reading order and table serialization.

In [65]:
REPEATED_LINE_RE = re.compile(

    r"(?im)^\s*(?:central motor vehicles rules,?\s*1989|cmvr\s*[-:]?\s*1989|page\s+\d+|\d+)\s*$"

)

AIS_SPACING_RE = re.compile(

    r"\bA\s*[-:]?\s*I\s*[-:]?\s*S\s*[-:]?\s*(?=\d)",

    re.IGNORECASE,

)





def normalize_pdf_text(text: str) -> str:

    """Repair common PDF artifacts while preserving legal identifiers."""



    value = unicodedata.normalize("NFKC", text)

    value = value.replace("\u00ad", "").replace("\ufeff", "").replace("\x0c", "\n")

    value = value.replace("−", "-").replace("–", "-").replace("—", "-")

    value = value.replace("“", '"').replace("”", '"').replace("’", "'")

    value = re.sub(r"(?<=\w)-\s*\n\s*(?=[a-z])", "", value)

    value = AIS_SPACING_RE.sub("AIS-", value)

    value = REPEATED_LINE_RE.sub("", value)

    value = re.sub(r"[ \t]+", " ", value)

    value = re.sub(r"\s*\n\s*", "\n", value)

    value = re.sub(r"\n{3,}", "\n\n", value)

    return value.strip()





normalization_probe = "A IS : 052\nCENTRAL MOTOR VEHICLES RULES, 1989\n115. Emis-\nsion"

assert normalize_pdf_text(normalization_probe) == "AIS-052\n115. Emission"

print("Normalization probe passed")

Normalization probe passed


## 4. Identify Chapters and Rule Boundaries



Use the repaired context already attached to each `RegulatoryChunk`. Sub-rules and provisos remain grouped under their base rule, so `Rule 115 > Sub-rule (2)` still produces the stable key `cmvr-1989:rule:115`.

In [66]:
RULE_CONTEXT_RE = re.compile(r"^Rule\s+(?P<number>\d+(?:-[A-Z])?)\b", re.IGNORECASE)

CHAPTER_CONTEXT_RE = re.compile(r"\bCHAPTER\s+(?P<number>[IVXLCDM]+|\d+)\b", re.IGNORECASE)





@dataclass(frozen=True)

class RuleCandidate:

    rule_number: str

    chapter_number: str

    source_pages: tuple[int, ...]

    source_text: str

    source_chunk_count: int



    @property

    def key_segment(self) -> str:

        return self.rule_number.lower()



    @property

    def canonical_key(self) -> str:

        return f"{DOCUMENT_ID}:rule:{self.key_segment}"



    @property

    def chapter_id(self) -> str:

        return f"{DOCUMENT_ID}:chapter:{self.chapter_number}"





def _rule_sort_key(rule_number: str) -> tuple[int, str]:

    match = re.fullmatch(r"(\d+)(?:-([A-Z]))?", rule_number)

    if not match:

        return (10**9, rule_number)

    return (int(match.group(1)), match.group(2) or "")





def group_chunks_by_rule(chunks: list[RegulatoryChunk]) -> tuple[list[RuleCandidate], list[dict[str, Any]]]:

    """Assign every continuation/annex chunk until the next genuine rule."""



    grouped: dict[str, dict[str, Any]] = defaultdict(

        lambda: {"texts": [], "seen": set(), "pages": set(), "chapter": None}

    )

    rejected: list[dict[str, Any]] = []

    active_rule: str | None = None

    active_chapter: str | None = None



    for chunk in chunks:

        chapter_match = CHAPTER_CONTEXT_RE.search(chunk.chapter or "")

        chunk_chapter = chapter_match.group("number").upper() if chapter_match else None

        rule_match = RULE_CONTEXT_RE.match(chunk.rule or "")



        if rule_match:

            active_rule = rule_match.group("number").upper()

            if not chunk_chapter:

                rejected.append(

                    {

                        "reason": "missing_chapter",

                        "rule": chunk.rule,

                        "pages": list(chunk.page_numbers),

                        "snippet": chunk.text[:240],

                    }

                )

                active_rule = None

                active_chapter = None

                continue

            active_chapter = chunk_chapter

            grouped[active_rule]["chapter"] = active_chapter

        elif active_rule and chunk_chapter and chunk_chapter != active_chapter:

            # A new chapter can have introductory text before its first rule.

            active_rule = None

            active_chapter = None



        if not active_rule:

            continue



        normalized_text = normalize_pdf_text(chunk.text)

        bucket = grouped[active_rule]

        bucket["pages"].update(chunk.page_numbers)

        if normalized_text and normalized_text not in bucket["seen"]:

            bucket["seen"].add(normalized_text)

            bucket["texts"].append(normalized_text)



    candidates = [

        RuleCandidate(

            rule_number=rule_number,

            chapter_number=bucket["chapter"],

            source_pages=tuple(sorted(bucket["pages"])),

            source_text="\n\n".join(bucket["texts"]),

            source_chunk_count=len(bucket["texts"]),

        )

        for rule_number, bucket in grouped.items()

        if bucket["texts"] and bucket["chapter"]

    ]

    candidates.sort(key=lambda candidate: _rule_sort_key(candidate.rule_number))

    return candidates, rejected





rule_candidates, boundary_errors = group_chunks_by_rule(regulatory_chunks)

print(f"Rule candidates: {len(rule_candidates):,}")

print(f"Boundary errors: {len(boundary_errors):,}")

Rule candidates: 177
Boundary errors: 0


## 5. Extract Rule Numbers, Titles, and Status



Derive a title fallback directly from the rule header. The LLM stage can clean OCR/amendment noise, while the canonical rule key remains deterministic and independent of the title or sub-rule text.

In [67]:
def deterministic_rule_title(candidate: RuleCandidate) -> str:

    number_pattern = re.escape(candidate.rule_number).replace(r"\-", r"\s*-?\s*")

    header_re = re.compile(

        rf"^\s*(?:\d+\s*\[\s*)?{number_pattern}\s*\.\s*(?P<body>.+)",

        re.IGNORECASE,

    )

    for block in candidate.source_text.split("\n\n"):

        match = header_re.match(block)

        if not match:

            continue

        title = match.group("body")

        title = re.split(

            r"(?:\d{1,4}\s*\[\s*)?\(1\)|\.\s*-|\s+-\s+",

            title,

            maxsplit=1,

        )[0]

        title = re.sub(r"^\d{1,4}\s*\[\s*", "", title)

        title = re.sub(r"\s*\d{1,4}\s*\[\s*$", "", title)

        title = title.strip(" .:-[]")

        if title:

            return title

    return f"Rule {candidate.rule_number}"





candidate_preview = pd.DataFrame(

    [

        {

            "canonicalKey": candidate.canonical_key,

            "ruleNumber": candidate.rule_number,

            "titleFallback": deterministic_rule_title(candidate),

            "chapterId": candidate.chapter_id,

            "pages": ", ".join(map(str, candidate.source_pages)),

            "chunks": candidate.source_chunk_count,

            "characters": len(candidate.source_text),

        }

        for candidate in rule_candidates[:20]

    ]

)



rule_115_candidate = next(candidate for candidate in rule_candidates if candidate.rule_number == "115")

assert rule_candidates[0].rule_number == "1" and rule_candidates[0].source_pages == (1,)

assert rule_115_candidate.chapter_id == f"{DOCUMENT_ID}:chapter:V"

rule_115_preview = {

    "canonicalKey": rule_115_candidate.canonical_key,

    "titleFallback": deterministic_rule_title(rule_115_candidate),

    "chapterId": rule_115_candidate.chapter_id,

    "sourcePages": list(rule_115_candidate.source_pages),

    "deterministicAIS": find_ais_codes(rule_115_candidate.source_text)

    if "find_ais_codes" in globals()

    else [],

}

display(candidate_preview)

display(pd.DataFrame([rule_115_preview]))

,canonicalKey,ruleNumber,titleFallback,chapterId,pages,chunks,characters
0,cmvr-1989:rule:1,1,Short title and commencement,cmvr-1989:chapter:I,1,1,569
1,cmvr-1989:rule:2,2,Definitions,cmvr-1989:chapter:I,"1, 2, 3, 4",40,7970
2,cmvr-1989:rule:3,3,General,cmvr-1989:chapter:II,"4, 5",6,1199
3,cmvr-1989:rule:4,4,Evidence as to the correctness of address and age,cmvr-1989:chapter:II,"5, 6",12,2053
4,cmvr-1989:rule:5,5,Medical certificate,cmvr-1989:chapter:II,6,3,876
5,cmvr-1989:rule:6,6,Exemption from production of medical certificate,cmvr-1989:chapter:II,6,1,487
6,cmvr-1989:rule:7,7,Affixing of photograph to medical certificate,cmvr-1989:chapter:II,6,1,374
7,cmvr-1989:rule:8,8,Minimum educational qualification for driving ...,cmvr-1989:chapter:II,6,1,136
8,cmvr-1989:rule:9,9,Educational qualifications for drivers of good...,cmvr-1989:chapter:II,"6, 7, 8",51,3994
9,cmvr-1989:rule:10,10,Application for learner's licence,cmvr-1989:chapter:II,"8, 9",3,527


,canonicalKey,titleFallback,chapterId,sourcePages,deterministicAIS
0,cmvr-1989:rule:115,"Emission of smoke, vapour, etc. from motor veh...",cmvr-1989:chapter:V,"[96, 97, 98, 99, 100, 101, 102, 103, 104, 105,...","[AIS-025, AIS-026, AIS-027, AIS-054, AIS-055]"


## 6. Extract AIS References and Descriptions



Inventory AIS identifiers deterministically, then ask OpenAI to clean the rule title and describe each referenced AIS from nearby paragraphs, clauses, footnotes, and serialized table rows. The LLM schema uses a list rather than a free-form dictionary because OpenAI Structured Outputs requires closed objects.

In [68]:
AIS_ID_RE = re.compile(

    r"\bA\s*[-:]?\s*I\s*[-:]?\s*S\s*[-:]?\s*(?P<number>\d{1,3})\b",

    re.IGNORECASE,

)





class ClosedExtractionModel(BaseModel):

    model_config = ConfigDict(extra="forbid", strict=True, str_strip_whitespace=True)





class AISReferenceExtraction(ClosedExtractionModel):

    code: str = Field(pattern=r"^AIS-\d{3}$")

    description: str = Field(min_length=1, max_length=1_000)





class RuleLLMExtraction(ClosedExtractionModel):

    canonical_title: str = Field(min_length=1, max_length=300)

    ais: list[AISReferenceExtraction]



    @model_validator(mode="after")

    def ais_codes_must_be_unique(self) -> "RuleLLMExtraction":

        codes = [reference.code for reference in self.ais]

        if len(codes) != len(set(codes)):

            raise ValueError("AIS codes must be unique within a rule")

        return self





def normalize_ais_code(value: str) -> str:

    match = AIS_ID_RE.search(value)

    if not match:

        raise ValueError(f"Invalid AIS identifier: {value!r}")

    return f"AIS-{int(match.group('number')):03d}"





def find_ais_codes(text: str) -> list[str]:

    return sorted({normalize_ais_code(match.group(0)) for match in AIS_ID_RE.finditer(text)})





def assert_openai_compatible_schema(model: type[BaseModel]) -> None:

    schema = to_strict_json_schema(model)



    def visit(node: object, path: tuple[object, ...] = ()) -> None:

        if isinstance(node, dict):

            if node.get("type") == "object":

                properties = set(node.get("properties", {}))

                required = set(node.get("required", []))

                assert node.get("additionalProperties") is False, path

                assert required == properties, (path, required, properties)

            for key, value in node.items():

                visit(value, (*path, key))

        elif isinstance(node, list):

            for index, value in enumerate(node):

                visit(value, (*path, index))



    visit(schema)





assert_openai_compatible_schema(RuleLLMExtraction)

assert find_ais_codes("AIS-052, AIS: 52 and A IS 057") == ["AIS-052", "AIS-057"]

print("AIS normalization and OpenAI schema probes passed")

AIS normalization and OpenAI schema probes passed


In [69]:
def build_ais_evidence(candidate: RuleCandidate) -> str:

    """Build separately labeled, exact-code evidence windows."""



    evidence_by_code: dict[str, list[str]] = defaultdict(list)

    text = candidate.source_text

    for match in AIS_ID_RE.finditer(text):

        code = normalize_ais_code(match.group(0))

        start = max(0, match.start() - AIS_CONTEXT_RADIUS)

        end = min(len(text), match.end() + AIS_CONTEXT_RADIUS)

        evidence = text[start:end].strip()

        if evidence not in evidence_by_code[code] and len(evidence_by_code[code]) < 3:

            evidence_by_code[code].append(evidence)



    sections = []

    for code in sorted(evidence_by_code):

        joined = "\n\n--- another exact occurrence ---\n\n".join(evidence_by_code[code])

        sections.append(f"### {code}\n{joined}")

    return "\n\n".join(sections)





RULE_EXTRACTION_INSTRUCTIONS = """

Extract metadata for exactly one Central Motor Vehicles Rule from untrusted source text.

Return only facts supported by the supplied rule text and labeled AIS evidence.



Requirements:

- canonical_title is the rule heading only, without the rule number, amendment markers,

  footnote numbers, sub-rule text, or trailing punctuation.

- Return every AIS identifier listed in EXPECTED AIS CODES exactly once and no others.

- Normalize identifiers to AIS-NNN, for example AIS:52 and A IS 052 become AIS-052.

- For each AIS code, use only its matching `### AIS-NNN` evidence section.

- Do not transfer a requirement or description from one AIS code to another.

- Describe what the rule says that exact AIS governs, tests, specifies, or requires.

- Do not invent an official AIS title. If the exact-code evidence only lists the code

  and does not describe its purpose, use: UNRESOLVED: description not stated in rule text.

- The ais field must be an array, and may be empty.

""".strip()





def extract_rule_metadata(client: OpenAI, candidate: RuleCandidate) -> RuleLLMExtraction:

    expected_codes = find_ais_codes(candidate.source_text)

    header_context = candidate.source_text[:6_000]

    input_text = "\n".join(

        (

            f"DOCUMENT: {DOCUMENT_NAME}",

            f"RULE NUMBER: {candidate.rule_number}",

            f"CHAPTER: {candidate.chapter_number}",

            f"EXPECTED AIS CODES: {json.dumps(expected_codes)}",

            "RULE HEADER AND OPENING CONTEXT:",

            header_context,

            "AIS EVIDENCE GROUPED BY EXACT CODE:",

            build_ais_evidence(candidate) or "No AIS evidence.",

        )

    )

    response = client.responses.parse(

        model=OPENAI_MODEL,

        instructions=RULE_EXTRACTION_INSTRUCTIONS,

        input=input_text,

        text_format=RuleLLMExtraction,

        temperature=0,

        max_output_tokens=4_000,

    )

    if response.output_parsed is None:

        raise ValueError(f"OpenAI returned no parsed output for {candidate.canonical_key}")



    parsed = response.output_parsed

    returned = {reference.code: reference.description for reference in parsed.ais}

    grounded_references = [

        AISReferenceExtraction(

            code=code,

            description=returned.get(

                code,

                "UNRESOLVED: description not stated in rule text",

            ),

        )

        for code in expected_codes

    ]

    return RuleLLMExtraction(

        canonical_title=parsed.canonical_title or deterministic_rule_title(candidate),

        ais=grounded_references,

    )





print("Exact-code AIS evidence extraction configured")

Exact-code AIS evidence extraction configured


## 7. Build Canonical Rule Documents



Build the requested document shape with complete normalized `ruleText`. Legacy checkpoint rows are backfilled from the corresponding `RuleCandidate.source_text` and the checkpoint is rewritten atomically, preserving all previously extracted AIS descriptions without repeating paid calls.

In [ ]:
class CMVRRuleDocument(BaseModel):

    model_config = ConfigDict(extra="forbid", populate_by_name=True, str_strip_whitespace=True)



    id: str = Field(alias="_id", pattern=rf"^{re.escape(DOCUMENT_ID)}:rule:\d+(?:-[a-z])?$")

    canonicalKey: str

    canonicalTitle: str = Field(min_length=1, max_length=300)

    chapterId: str = Field(pattern=rf"^{re.escape(DOCUMENT_ID)}:chapter:[IVXLCDM]+$")

    documentId: str

    ruleNumber: str = Field(pattern=r"^\d+(?:-[A-Z])?$")

    ruleText: str = Field(min_length=1, max_length=5_000_000)

    status: str = Field(pattern=r"^active$")

    AIS: dict[str, str]



    @model_validator(mode="after")

    def canonical_identifiers_must_agree(self) -> "CMVRRuleDocument":

        expected_key = f"{self.documentId}:rule:{self.ruleNumber.lower()}"

        if self.id != expected_key or self.canonicalKey != expected_key:

            raise ValueError("_id, canonicalKey, documentId, and ruleNumber disagree")

        if self.documentId != DOCUMENT_ID:

            raise ValueError(f"documentId must be {DOCUMENT_ID}")

        if any(not re.fullmatch(r"AIS-\d{3}", code) for code in self.AIS):

            raise ValueError("Every AIS key must match AIS-NNN")

        return self





def build_rule_document(

    candidate: RuleCandidate,

    extraction: RuleLLMExtraction,

) -> CMVRRuleDocument:

    ais_map = {reference.code: reference.description for reference in extraction.ais}

    return CMVRRuleDocument.model_validate(

        {

            "_id": candidate.canonical_key,

            "canonicalKey": candidate.canonical_key,

            "canonicalTitle": extraction.canonical_title,

            "chapterId": candidate.chapter_id,

            "documentId": DOCUMENT_ID,

            "ruleNumber": candidate.rule_number,

            "ruleText": candidate.source_text,

            "status": "active",

            "AIS": dict(sorted(ais_map.items())),

        }

    )





def load_checkpoint(

    path: Path,

    candidates: dict[str, RuleCandidate],

) -> tuple[dict[str, dict[str, Any]], list[dict[str, Any]], int]:

    """Load and backfill legacy rows without repeating OpenAI extraction."""



    loaded: dict[str, dict[str, Any]] = {}

    errors: list[dict[str, Any]] = []

    migrated = 0

    if not path.exists():

        return loaded, errors, migrated



    for line_number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), start=1):

        if not line.strip():

            continue

        try:

            payload = json.loads(line)

            canonical_key = payload.get("canonicalKey") or payload.get("_id")

            candidate = candidates.get(canonical_key)

            if candidate is None:

                raise ValueError(f"No current RuleCandidate for {canonical_key!r}")

            if payload.get("ruleText") != candidate.source_text:

                payload["ruleText"] = candidate.source_text

                migrated += 1

            document = CMVRRuleDocument.model_validate(payload)

            loaded[document.canonicalKey] = document.model_dump(by_alias=True)

        except (json.JSONDecodeError, ValidationError, ValueError) as error:

            errors.append(

                {"line": line_number, "error": str(error), "payload": line[:500]}

            )

    return loaded, errors, migrated





def append_checkpoint(path: Path, document: CMVRRuleDocument) -> None:

    with path.open("a", encoding="utf-8") as checkpoint:

        checkpoint.write(document.model_dump_json(by_alias=True) + "\n")





def rewrite_checkpoint(path: Path, documents: dict[str, dict[str, Any]]) -> None:

    """Atomically compact migrated/current checkpoint records."""



    temporary_path = path.with_suffix(path.suffix + ".tmp")

    ordered_keys = sorted(

        documents,

        key=lambda value: _rule_sort_key(value.rsplit(":", 1)[-1].upper()),

    )

    with temporary_path.open("w", encoding="utf-8") as checkpoint:

        for key in ordered_keys:

            document = CMVRRuleDocument.model_validate(documents[key])

            checkpoint.write(document.model_dump_json(by_alias=True) + "\n")

    temporary_path.replace(path)





selected_candidates = rule_candidates[:MAX_RULES] if MAX_RULES else rule_candidates

selected_keys = {candidate.canonical_key for candidate in selected_candidates}

candidate_by_key = {candidate.canonical_key: candidate for candidate in rule_candidates}

checkpoint_documents, checkpoint_errors, migrated_checkpoint_rows = load_checkpoint(

    CHECKPOINT_PATH,

    candidate_by_key,

)

if migrated_checkpoint_rows and not checkpoint_errors:

    rewrite_checkpoint(CHECKPOINT_PATH, checkpoint_documents)

    print(f"Backfilled ruleText and rewrote {migrated_checkpoint_rows:,} checkpoint rows")



llm_errors: list[dict[str, Any]] = []

pending_ais_candidates: list[dict[str, Any]] = []

deterministic_documents_created = 0



openai_client = None

if RUN_LLM_EXTRACTION:

    if not OPENAI_API_KEY:

        raise RuntimeError("Set OPENAI_API_KEY before enabling RUN_LLM_EXTRACTION")

    openai_client = OpenAI(

        api_key=OPENAI_API_KEY,

        timeout=OPENAI_TIMEOUT_SECONDS,

        max_retries=OPENAI_MAX_RETRIES,

    )



for index, candidate in enumerate(selected_candidates, start=1):

    if candidate.canonical_key in checkpoint_documents:

        continue



    ais_codes = find_ais_codes(candidate.source_text)

    if ais_codes and not RUN_LLM_EXTRACTION:

        pending_ais_candidates.append(

            {

                "canonicalKey": candidate.canonical_key,

                "ruleNumber": candidate.rule_number,

                "AIS": ais_codes,

                "sourcePages": list(candidate.source_pages),

            }

        )

        continue



    try:

        if ais_codes:

            extraction = extract_rule_metadata(openai_client, candidate)

        else:

            extraction = RuleLLMExtraction(

                canonical_title=deterministic_rule_title(candidate),

                ais=[],

            )

            deterministic_documents_created += 1

        document = build_rule_document(candidate, extraction)

        append_checkpoint(CHECKPOINT_PATH, document)

        checkpoint_documents[document.canonicalKey] = document.model_dump(by_alias=True)

    except Exception as error:

        LOGGER.exception("Rule extraction failed for %s", candidate.canonical_key)

        llm_errors.append(

            {

                "canonicalKey": candidate.canonical_key,

                "ruleNumber": candidate.rule_number,

                "sourcePages": list(candidate.source_pages),

                "error": f"{type(error).__name__}: {error}",

                "snippet": candidate.source_text[:500],

            }

        )



    if index % 10 == 0 or index == len(selected_candidates):

        print(f"Processed {index:,}/{len(selected_candidates):,} rule candidates")



rule_documents = [

    checkpoint_documents[key]

    for key in sorted(

        selected_keys & checkpoint_documents.keys(),

        key=lambda value: _rule_sort_key(value.rsplit(":", 1)[-1].upper()),

    )

]

missing_document_keys = selected_keys - checkpoint_documents.keys()

print(f"Selected rule candidates: {len(selected_candidates):,}")

print(f"Deterministic no-AIS documents created: {deterministic_documents_created:,}")

print(f"Valid selected checkpoint documents: {len(rule_documents):,}")

print(f"AIS-bearing rules pending OpenAI extraction: {len(pending_ais_candidates):,}")

print(f"Checkpoint errors: {len(checkpoint_errors):,}")

print(f"LLM errors in this run: {len(llm_errors):,}")

print(f"Selected documents still missing: {len(missing_document_keys):,}")

if pending_ais_candidates:

    print("Enable RUN_LLM_EXTRACTION and rerun this cell to complete AIS-bearing rules.")



detected_rule_115_ais = find_ais_codes(rule_115_candidate.source_text)

rule_115_example = next(

    (document for document in rule_documents if document["ruleNumber"] == "115"),

    {

        "_id": rule_115_candidate.canonical_key,

        "canonicalKey": rule_115_candidate.canonical_key,

        "canonicalTitle": deterministic_rule_title(rule_115_candidate),

        "chapterId": rule_115_candidate.chapter_id,

        "documentId": DOCUMENT_ID,

        "ruleNumber": rule_115_candidate.rule_number,

        "ruleText": rule_115_candidate.source_text,

        "status": "active",

        "AIS": {

            code: "UNRESOLVED: enable LLM extraction to generate a source-grounded description"

            for code in detected_rule_115_ais

        },

    },

)

CMVRRuleDocument.model_validate(rule_115_example)

print(json.dumps({**rule_115_example, "ruleText": rule_115_example["ruleText"][:500] + "..."}, indent=2, ensure_ascii=False))

Selected rule candidates: 177
Deterministic no-AIS documents created: 0
Valid selected checkpoint documents: 177
AIS-bearing rules pending OpenAI extraction: 0
Checkpoint errors: 0
LLM errors in this run: 0
Selected documents still missing: 0
{
  "_id": "cmvr-1989:rule:115",
  "canonicalKey": "cmvr-1989:rule:115",
  "canonicalTitle": "Emission of smoke, vapour, etc. from motor vehicles",
  "chapterId": "cmvr-1989:chapter:V",
  "documentId": "cmvr-1989",
  "ruleNumber": "115",
  "ruleText": "115. Emission of smoke, vapour, etc. from motor vehicles.258 [(1) Every motor vehicle other than motor cycles of engine capacity not exceeding 70 cc, manufactured prior to the first day of March 1990, shall be maintained in such condition and shall be so driven so as to comply with the standards prescribed in these rules.]\n\n259 [(2) On and after 1st October, 2004, every motor vehicle operating on-\n\n(i) Petrol/CNG/LPG shall comply with the idling emission standards for Carbon monoxide (CO) and Hy

## 8. Validate and Review Extracted Documents



Validate canonical identifiers and uniqueness, then review valid records, duplicates, missing titles, and unresolved AIS descriptions before enabling MongoDB writes.

In [ ]:
validation_errors: list[dict[str, Any]] = []valid_documents: list[dict[str, Any]] = []for payload in rule_documents:    try:        validated = CMVRRuleDocument.model_validate(payload)        valid_documents.append(validated.model_dump(by_alias=True))    except ValidationError as error:        validation_errors.append(            {                "canonicalKey": payload.get("canonicalKey"),                "error": str(error),            }        )records_df = pd.DataFrame(valid_documents)if records_df.empty:    duplicate_rules_df = pd.DataFrame(columns=["ruleNumber", "count"])    missing_titles_df = pd.DataFrame(columns=["canonicalKey", "ruleNumber"])    missing_rule_text_df = pd.DataFrame(columns=["canonicalKey", "ruleNumber"])else:    records_df["ruleTextCharacters"] = records_df["ruleText"].str.len()    duplicate_counts = records_df["ruleNumber"].value_counts()    duplicate_rules_df = duplicate_counts[duplicate_counts > 1].rename("count").reset_index()    missing_titles_df = records_df[records_df["canonicalTitle"].str.strip().eq("")][        ["canonicalKey", "ruleNumber"]    ]    missing_rule_text_df = records_df[records_df["ruleText"].str.strip().eq("")][        ["canonicalKey", "ruleNumber"]    ]valid_keys = {document["canonicalKey"] for document in valid_documents}missing_documents_df = pd.DataFrame(    [        {            "canonicalKey": candidate.canonical_key,            "ruleNumber": candidate.rule_number,            "AIS": find_ais_codes(candidate.source_text),            "sourcePages": list(candidate.source_pages),        }        for candidate in selected_candidates        if candidate.canonical_key not in valid_keys    ])text_mismatches_df = pd.DataFrame(    [        {            "canonicalKey": document["canonicalKey"],            "storedCharacters": len(document["ruleText"]),            "sourceCharacters": len(candidate_by_key[document["canonicalKey"]].source_text),        }        for document in valid_documents        if document["canonicalKey"] in candidate_by_key        and document["ruleText"] != candidate_by_key[document["canonicalKey"]].source_text    ])extraction_complete = (    missing_documents_df.empty    and missing_rule_text_df.empty    and text_mismatches_df.empty    and not validation_errors    and not llm_errors)unresolved_ais_rows = [    {        "canonicalKey": document["canonicalKey"],        "ruleNumber": document["ruleNumber"],        "aisCode": code,        "description": description,        "sourcePages": list(candidate_by_key[document["canonicalKey"]].source_pages)        if document["canonicalKey"] in candidate_by_key        else [],    }    for document in valid_documents    for code, description in document["AIS"].items()    if description.startswith("UNRESOLVED:")]unresolved_ais_df = pd.DataFrame(unresolved_ais_rows)assert not records_df.get("_id", pd.Series(dtype=str)).duplicated().any(), "Duplicate _id values"assert duplicate_rules_df.empty, "Duplicate rule numbers must be resolved before ingestion"assert missing_rule_text_df.empty, "Every selected rule requires ruleText"assert text_mismatches_df.empty, "Stored ruleText must match normalized source text"print(f"Valid documents: {len(valid_documents):,}/{len(selected_candidates):,}")print(f"Missing selected documents: {len(missing_documents_df):,}")print(f"Missing rule text: {len(missing_rule_text_df):,}")print(f"Rule text mismatches: {len(text_mismatches_df):,}")print(f"Validation errors: {len(validation_errors):,}")print(f"Missing titles: {len(missing_titles_df):,}")print(f"Unresolved AIS descriptions: {len(unresolved_ais_df):,}")print(f"Extraction complete: {extraction_complete}")display(records_df[["canonicalKey", "ruleNumber", "canonicalTitle", "chapterId", "ruleTextCharacters"]].head(20))display(missing_documents_df.head(50))display(missing_rule_text_df)display(text_mismatches_df)display(duplicate_rules_df)display(missing_titles_df)display(unresolved_ais_df.head(50))

Valid documents: 177/177
Missing selected documents: 0
Missing rule text: 0
Rule text mismatches: 0
Validation errors: 0
Missing titles: 0
Unresolved AIS descriptions: 0
Extraction complete: True


,canonicalKey,ruleNumber,canonicalTitle,chapterId,ruleTextCharacters
0,cmvr-1989:rule:1,1,Short title and commencement,cmvr-1989:chapter:I,569
1,cmvr-1989:rule:2,2,Definitions,cmvr-1989:chapter:I,7970
2,cmvr-1989:rule:3,3,General,cmvr-1989:chapter:II,1199
3,cmvr-1989:rule:4,4,Evidence as to the correctness of address and age,cmvr-1989:chapter:II,2053
4,cmvr-1989:rule:5,5,Medical certificate,cmvr-1989:chapter:II,876
5,cmvr-1989:rule:6,6,Exemption from production of medical certificate,cmvr-1989:chapter:II,487
6,cmvr-1989:rule:7,7,Affixing of photograph to medical certificate,cmvr-1989:chapter:II,374
7,cmvr-1989:rule:8,8,Minimum educational qualification for driving ...,cmvr-1989:chapter:II,136
8,cmvr-1989:rule:9,9,Educational qualifications for drivers of good...,cmvr-1989:chapter:II,3994
9,cmvr-1989:rule:10,10,Application for learner's licence,cmvr-1989:chapter:II,527


""


,canonicalKey,ruleNumber


""


,ruleNumber,count


,canonicalKey,ruleNumber


""


## 9. Insert or Update Rules in MongoDB



Perform idempotent bulk upserts by `_id`. The persisted document keeps the requested fields and adds `createdAt`, `updatedAt`, and `source` metadata for lineage. This cell does nothing unless `WRITE_TO_MONGODB=True`.

In [72]:
def build_mongo_update(document: dict[str, Any], now: datetime) -> UpdateOne:

    candidate = candidate_by_key.get(document["canonicalKey"])

    source = {

        "documentName": DOCUMENT_NAME,

        "pdfFile": PDF_PATH.name,

        "pages": list(candidate.source_pages) if candidate else [],

        "chunkCount": candidate.source_chunk_count if candidate else 0,

        "extractionModel": OPENAI_MODEL if document["AIS"] else "deterministic-header",

    }

    stored_fields = {**document, "source": source, "updatedAt": now}

    stored_fields.pop("_id", None)

    return UpdateOne(

        {"_id": document["_id"]},

        {

            "$set": stored_fields,

            "$setOnInsert": {"createdAt": now},

        },

        upsert=True,

    )





mongo_summary: dict[str, Any] | None = None

if WRITE_TO_MONGODB:

    if not MONGODB_URI:

        raise RuntimeError("Set MONGODB_URI before enabling WRITE_TO_MONGODB")

    if not valid_documents:

        raise RuntimeError(

            "No validated documents exist. Rerun Cell 16 to build/checkpoint documents, "

            "then rerun Cell 18 before this MongoDB cell."

        )

    if not extraction_complete:

        raise RuntimeError(

            f"Extraction is incomplete: {len(missing_documents_df)} selected rules are missing. "

            "Set RUN_LLM_EXTRACTION=True in Cell 3, rerun Cell 16, then rerun Cell 18."

        )



    mongo_client = MongoClient(

        MONGODB_URI,

        appname="cmvr-rule-ais-notebook",

        serverSelectionTimeoutMS=10_000,

        tz_aware=True,

    )

    try:

        mongo_client.admin.command("ping")

        collection = mongo_client[MONGODB_DATABASE][MONGODB_COLLECTION]

        collection.create_index(

            [("canonicalKey", ASCENDING)],

            unique=True,

            name="canonical_key_unique",

        )

        collection.create_index(

            [("documentId", ASCENDING), ("ruleNumber", ASCENDING)],

            unique=True,

            name="document_rule_unique",

        )

        collection.create_index([("chapterId", ASCENDING)], name="chapter_lookup")

        collection.create_index([("AIS.$**", ASCENDING)], name="ais_wildcard")



        now = datetime.now(timezone.utc)

        totals = Counter()

        for start in range(0, len(valid_documents), MONGO_BATCH_SIZE):

            batch = valid_documents[start : start + MONGO_BATCH_SIZE]

            operations = [build_mongo_update(document, now) for document in batch]

            try:

                result = collection.bulk_write(operations, ordered=False)

                totals["matched"] += result.matched_count

                totals["modified"] += result.modified_count

                totals["upserted"] += result.upserted_count

            except BulkWriteError as error:

                LOGGER.error("MongoDB batch failed at offset %d: %s", start, error.details)

                raise

        mongo_summary = dict(totals)

        print("MongoDB upsert summary:", mongo_summary)

    finally:

        mongo_client.close()

else:

    print("MongoDB writes are disabled. Review Section 8, then set WRITE_TO_MONGODB=True.")

MongoDB upsert summary: {'matched': 0, 'modified': 0, 'upserted': 177}


## 10. Verify Stored MongoDB Records



Query counts, indexes, and Rule 115 after an enabled write. The verification cell is read-only and remains gated by `WRITE_TO_MONGODB` to avoid accidental connections.

In [73]:
if WRITE_TO_MONGODB:

    verification_client = MongoClient(

        MONGODB_URI,

        serverSelectionTimeoutMS=10_000,

        tz_aware=True,

    )

    try:

        collection = verification_client[MONGODB_DATABASE][MONGODB_COLLECTION]

        stored_count = collection.count_documents({"documentId": DOCUMENT_ID})

        missing_stored_text_count = collection.count_documents(

            {

                "documentId": DOCUMENT_ID,

                "$or": [

                    {"ruleText": {"$exists": False}},

                    {"ruleText": ""},

                ],

            }

        )

        indexes = collection.index_information()

        stored_rule_115 = collection.find_one({"_id": f"{DOCUMENT_ID}:rule:115"})

        print(f"Stored {DOCUMENT_ID} rules: {stored_count:,}")

        print(f"Stored rules missing ruleText: {missing_stored_text_count:,}")

        print("Indexes:", sorted(indexes))

        assert stored_count == len(valid_documents)

        assert missing_stored_text_count == 0



        printable_rule_115 = dict(stored_rule_115 or {})

        if printable_rule_115.get("ruleText"):

            printable_rule_115["ruleText"] = printable_rule_115["ruleText"][:1_000] + "..."

        print(json.dumps(printable_rule_115, default=str, indent=2, ensure_ascii=False))

        if any(document["ruleNumber"] == "115" for document in valid_documents):

            assert stored_rule_115 is not None, "Rule 115 was expected but not found"

            assert stored_rule_115["chapterId"] == f"{DOCUMENT_ID}:chapter:V"

            assert stored_rule_115["status"] == "active"

            assert isinstance(stored_rule_115["AIS"], dict)

            assert stored_rule_115["ruleText"] == rule_115_candidate.source_text

            assert stored_rule_115["ruleText"].startswith("115.")

            print(f"Rule 115 text characters stored: {len(stored_rule_115['ruleText']):,}")

            if "AIS-052" in detected_rule_115_ais:

                assert stored_rule_115["AIS"].get("AIS-052"), "AIS-052 needs a description"

            else:

                print(

                    "AIS-052 is not cited in this Rule 115 source; stored actual citations:",

                    sorted(stored_rule_115["AIS"]),

                )

    finally:

        verification_client.close()

else:

    print("Verification skipped because WRITE_TO_MONGODB=False.")

Stored cmvr-1989 rules: 177
Stored rules missing ruleText: 0
Indexes: ['_id_', 'ais_wildcard', 'canonical_key_unique', 'chapter_lookup', 'document_rule_unique']
{
  "_id": "cmvr-1989:rule:115",
  "AIS": {
    "AIS-025": "Specifies the type approval requirements and performance tests for LPG kits and vehicles, including mass emission tests, engine performance tests, constant speed fuel consumption tests, and safety checks for LPG kit components and installation. It also details responsibilities for type approval and certification timelines.",
    "AIS-026": "Details the test procedures and safety guidelines for LPG vehicles, kit components, and installation, including mass emission tests, engine performance tests, gradeability tests, constant speed fuel consumption tests, electro-magnetic interference tests, range tests, and cooling performance tests. It assigns responsibility for type approval and certification timelines.",
    "AIS-027": "Provides additional test procedures and safety

## 11. Export Extraction Errors, Rule-Text Issues, and Unmatched AIS References



Export malformed boundaries, LLM/checkpoint failures, missing or mismatched rule text, duplicate rules, missing titles, and unresolved AIS descriptions with source pages/snippets for iterative review.

In [74]:
REVIEW_DIR.mkdir(parents=True, exist_ok=True)



review_errors = [

    *({"category": "boundary", **error} for error in boundary_errors),

    *({"category": "llm", **error} for error in llm_errors),

    *({"category": "validation", **error} for error in validation_errors),

    *({"category": "checkpoint", **error} for error in checkpoint_errors),

]



(REVIEW_DIR / "extraction_errors.json").write_text(

    json.dumps(review_errors, indent=2, ensure_ascii=False, default=str),

    encoding="utf-8",

)

unresolved_ais_df.to_csv(REVIEW_DIR / "unresolved_ais.csv", index=False)

duplicate_rules_df.to_csv(REVIEW_DIR / "duplicate_rules.csv", index=False)

missing_titles_df.to_csv(REVIEW_DIR / "missing_titles.csv", index=False)

missing_rule_text_df.to_csv(REVIEW_DIR / "missing_rule_text.csv", index=False)

text_mismatches_df.to_csv(REVIEW_DIR / "rule_text_mismatches.csv", index=False)



print(f"Review artifacts written to: {REVIEW_DIR.resolve()}")

print(f"Extraction errors: {len(review_errors):,}")

print(f"Missing rule text: {len(missing_rule_text_df):,}")

print(f"Rule text mismatches: {len(text_mismatches_df):,}")

print(f"Unresolved AIS references: {len(unresolved_ais_df):,}")

Review artifacts written to: /Users/utsav.talwar/Desktop/UST/ust-demo/cmvr_rule_review
Extraction errors: 0
Missing rule text: 0
Rule text mismatches: 0
Unresolved AIS references: 0
